# Trabajo final · Data Scientist · Sembra

**Microcréditos** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿A qué solicitudes conviene aprobarles el crédito?**

**Decisión que habilita:** La política de aprobación: el puntaje de corte y el monto máximo.

**Tipo de problema:** Clasificación desbalanceada, con costo asimétrico.

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNqdWUtz20YSvvtXTGHXW0BMQQApybIU5RJXqvYgX+w9pFgq1AgYykgADDQAGD9W+T855OTbXvXH9uuewYuklK11mRQw09PT7xfz"
    "stamFVVX1p+FbERVL0QtqwyP+F9nL15kaiPuVKWMNEkmW934jSrzopBXy2h5tlyIrTKZzK5+kkWjgosXAv+MuAKm0ACRLkNgkF3R"
    "Jqa6688Gl+IdYOKTKIouhfrUGpmmua6wVmfhh7xUTSvL2vdwx+lRvDxaxV7AqHXa1YAyYfpR56ny116qS2Vw2FsIr1Fmm+O5oZdU"
    "V01rOkZM77ikaohdRW+qrAulE1UlRhWyh0k7VbWyNrrOQQGtZDpVVZrzcyOLLvNuFuIdpHS1DsF9GJ/QFz/hs+xfojP+urFU59XG"
    "iiRv8sonHhZiTvnzxO6QdQP5KQjd4kyLvPZNWGlTysJfnS9EDFLeBQuxjBbidRSERndV5uOQrNr8bnbqTpal9JchIJfndEp8J/xY"
    "HIkoXJ3iGZQDEfBBVQMix9OdUY222NSnCQlxHJ6eMor43KJY4PmE0YOIL7qScxVWPZ9NZ1g3jX3V/QNU0BrNkgeCVJo7zRhqnTeN"
    "rvw4jN0WC4r3rPX54Oh74UfhuaVo2TPFPGRVbvEoU3atbCF8H9xInL1TfrxMYJ/0gQSGZ/HKsrGpZHrYXqMY/70A92Gz1UmL/UwV"
    "rfR91tp3YnV2CkqCUDbt51r5edUGQGtCPKg7ZRofNwLGarGr8vbKe+scoNGFvfWtbOVPRpbK/+rlWYJ1WH4LDSvvQkyYABLgjgOy"
    "5SrHHr7xvFHpR5mAh7zMITONDeKIpA7ztA5xIaypemw3HRGfFPJWG1kkYFk1gOAtgDhzwHrVdLLAjlvBHmkcC/QHbxA2dCabRCas"
    "SezwX7J6kKISq8TkVlZYz+mkXXmwErimyHHGkeO+wwFnzxrQrarw8dcj+yTAwdLgttfQyruAHMiEzcdusymUz1gsbpbKIa2uBq2+"
    "2tPqXG+vV/i6nurtUrwHSigozAudrvm6mxCyUW2SV5n65Gfw7KsPplOWilJDIS6GssftediSIk0EN6OrAnEsYsiDfDdmudSF/KJn"
    "PoZw9IasGB+cic/hStc2iNE7x60VPcT2y8WttNPsTZYgYCfHPrborR/SroU6Fu/DHSMItwiYqmFUfxO3nXn8xs+s5qmTXpOTgsnf"
    "Piqj/PfhYII2YO6Eyr0YPQueNwFJ5jXL543zmVQbdShYnsXQ1RvWF+yCY8P7cNc+HRfkRBTOgKTQd3ENMp9yi/4E1LIkpazgzRHf"
    "tBNDM9VRqj0YQi0DrF+IGhwihzLsNOyd0f4l9GNkUhu1PSRUZhnhPAKLvhXFEdFDZrNcEl1RGPFtJ4FTVaZSCP7xWyX8WhePf7Z5"
    "KrEoCimQN6FiaSGNuivIAvxBd6zahZP4D1fi9CRa7JBEmQU2+w/hWwP6/oquXlqMNVyEbd//nZEz3O75CAiQzGACt8QwE/Fvd9Ta"
    "raxkYcVqybJ7FHz4IYFVJZu8oKSyGKF271mesnGVuc1Csmt1KUkWXtBLimiQt3mRU2A3CrdCTKQOiFrfQgZMpcx0Iz6LSo+vVGcB"
    "qtaZMgKRJDcChy15FvkXMHC0Ir29EqvwBHZk5fVK+EeHBb4Qp2cRWfKZ1e7rU7I+zn0w3jCy1sv24J+eUEJ8Bg0hOIEAoL93ulIU"
    "+GCPQDEaG63EhHU0T1oiN/ofvON79o5XfaVxyOv/ytN7VENmfx8eTCGjCw8exmbP4dPaHSUVcIza51XvjUdfAudbe25V8yHKD5me"
    "2hmrl0zFqplpNpRTvtALsOVZcwCeojZXFqSermy60u4glB3FvalxmG3YDWFNqXn8I8tRj48mZTNYXnCUWN+Qh+zdFayjGwunjciR"
    "owFki3b6R7kUucznAL/OKU1uUf1ilRMjVmz6+zDkvkx+bq5WnHpmNUfFNQcFLsMZhLBO8iTCTQkVoySsAoYMRL5hUdMdCq2ExTBQ"
    "RuT+SuTaG6pgJJqL0Q12AU8hh6+8gHtxkQhq38kPM2BCP+wTf+tfD/NFRO+WyhyYUS8T2f7ZHtUsicFWAnaf5Xng4sWE3P7eHybt"
    "z3NEs15DWdcK2cOHGeHKhWV60SNb8HnEW04xbC8Mtezvdya0Uz8y6gXkVnRl1VytqZxMjSLrIvutjE74oDeUjXRfXzcOi3Q3vXGx"
    "4E5M64hm7bYACHP1bqbGaSHCERP8tK0kSdBtTdByxnKpPW876VrHAdnvzq0obk3COynqmWh/wtEeJXnO5DEniHRvIi945hRVEBZw"
    "FbG0OLx4vcRdWd5lqnmubEdbuVO0X/dF+15tjzJrtuSC26CHKUpeGXTSH8qo5uYlyojk61auVM/zK/UKCG55RYDTXlvWOF4kmUpg"
    "s7fyl6F37Up+3OZbWIWNe1v1MU+7gtc1Mm3jjUUnNc3Lsd6kD9VtHifuRJv8TlXzq2H5tkNMO4M+3AbWjTJ5pid4CduKcDM2G5+B"
    "xz4McugjppX5LBI7qWC5kpUTf+8L+9Bwwymspd4WT9w+8QJR3VspVZStMtTTDGuurUF5rP8fE1lzxruhWyh5J4SH0NPLuMeVZiLT"
    "Nt/KpIWuqUOz5acDcoVpFPQapXye4DsvdZOclcROvzziHYoB7tOGtx7npMF96PPY0dGRgBpzqgysj7juKNX1Z5pSoIuglSe6iUvR"
    "zfv7S9F+al0NXdbtZ2rzMrr2St/+otI2mOe7vi8cE8iW89MdQiU6Sjx3eBo2gZuCOzKg93fxdXuxCKPNg4fuDY4Cw/QWZImhx/nr"
    "3lW3NhX4G++rlem2b9AefvVGuBHsX+/fAjUBAcrdMIE7H/FtL8Ll7PqQrl9Mr59Dz1G9ObObMMhK0DSP4pQLVOu9Fp64BvtWfBWr"
    "ZHdqcCk2y71pC4XSS0K4N2OYhfwNEummCrM2REm3oXba915mxy/L45c/U8jd2/v56GV59DLri28jEVMOXB5FJ7g9pFabQQ4NO2aE"
    "DAgYPKTqK7CoKCNE8TH+02CHRB0vj6PT4/j89ekwkGTBjLWrtdLe8p0JooyM56T2Qh+nLTOaAM7ZZcnxBdSbsEPuN75Oe9SQTDDN"
    "TO7IcnJEZtnOAbAgvEOnzsdTLsrnX9T+bTodDMZKeFf4p8+NixyLCJe2/eVZLiQuaRjrb1AGXVF3txAWaYKw3Souw/rBcTBECRxe"
    "7yZHh7+f/djiiP/gU6jKx6FgOHvA3LGxF3Z2jGRAM1jI6EyXwmV5O43y1zA+wKJRmbIYP8Mfh/TnZkNNaqNlX1UM8mjS9V4BcNjM"
    "rwfSmzTcOfK0O/4l7I573veToZHEzWrqAyTIeyfGE3tkfaC+nMcM2MZ9uAf0NNV/DTySfTkSwLUs3Xw/LUmfOGjTN4Ap9Q7M3s79"
    "g7i9ddxG8Txb00WR9Qj6hQVvX72JWVNZRlFsonVO8DwLc2Xb/Sxruxrg9sFNadrOWBNtXrx4+88ffzx0w4EpMlYgsHyTpyiWDPVw"
    "ylRcz9lJMv040pUkU+pKLTCc3js8XrZrBDlZ3pk3e6a7NYxtUl8+M332+EGoCnmtPyJQ5XS2QtwfSvdLwi3RqKtAvLAFvxtU818i"
    "ApDElfv558DgelgT0g6xvWem2AgUbvTpfqYY9h4Wfb2xo+S9AnBHJXYmNwIc6he8+y5//KMiQBrd4fjjN+9gt+Blj3/KQzgPtA92"
    "TdQoja3sdvoIKobZGgif6zsZWPKPc2N74fEc7L57/EOgMqnBn7ffCHg1Mfv4rSLiCnVnORjq+2HeIrQYxy2H6n1Htm616bu8WYW/"
    "J99imLTMDu2V+l76+K3UxEKry8dvJMJhiuo90QL0izRond2kWFs94MQ6BvF+3SF7OAqTvu8gR4WzlUrVvIMHZPX4n1JZH3umq7+g"
    "/u2pJr+3FJBJd5HapFPJtPknHLIYrHS4bTYFGC3Jvk+Z3Y1n+94wvAy9idX7rBXy6g7e9otiGfOvEMy8k5j3VHdkV8WWTLBV9KMh"
    "tfbZMIc80CCN63QDml5JsYndwEapM2Ej117nxLGh22o7bgTkzmHv4eHFfwHGN/cc"
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "02_Sembra"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Llevar los ingresos a una sola unidad y moneda, unificar fechas, resolver solicitantes duplicados y documentar cada decisión.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Distribución de los ingresos y tasa de mora por segmento, con su intervalo de confianza.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Regresión logística, árboles, Random Forest y XGBoost; umbral y métricas según el costo de cada tipo de error.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red neuronal tabular contra la mejor referencia: ¿mejora lo bastante para justificar su costo y su opacidad?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Un tablero con la mora por segmento, la tasa de aprobación y el efecto del umbral.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada del servicio de scoring, versión del modelo y plan de monitoreo.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** La política de aprobación, el costo de equivocarse y una explicación global y una local, sin presentarlas como causalidad.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**